<a href="https://colab.research.google.com/github/MengOonLee/LLM/blob/main/References/LangChain/LangChainAcademy/LangChain/Foundation/CreateAgent/ipynb/1.4_multimodal_messages.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Text input

In [ ]:
%%bash
pip install --no-cache-dir -qU \
    langchain langgraph langchain-core \
    langchain-community langchain-openai

In [1]:
import warnings
warnings.filterwarnings("ignore")
import dotenv

_ =  dotenv.load_dotenv(dotenv_path=".env", override=True)

In [2]:
import os
import time
from langchain import chat_models, agents, messages

# Initialize using the OpenAI provider with a custom base_url
llm = chat_models.init_chat_model(
    model="gpt-oss:120b-cloud",
    model_provider="openai",
    base_url="https://ollama.com/v1",
    api_key=os.environ["OLLAMA_API_KEY"],
    temperature=0
)

agent = agents.create_agent(model=llm,
    system_prompt="""
        You are a science fiction writer,
        create a capital city at the users request.
    """
)

start_time = time.time()
question = messages.HumanMessage(content=[{
    "type": "text", "text": "What is the capital of The Moon?"
}])
response = agent.invoke(input={"messages": [question]})
end_time = time.time() - start_time
print(f"Time taken: %.2fs seconds"%(end_time))

for m in response["messages"]:
    m.pretty_print()

Time taken: 9.76s seconds
================================ Human Message =================================

[{'type': 'text', 'text': 'What is the capital of The Moon?'}]
================================== Ai Message ==================================

**The Moon’s Capital: Selene‑Polis**

---

### Overview
* **Name:** Selene‑Polis (often simply called *Selene*)
* **Location:** Inside the rim of the ancient, multi‑kilometer‑wide **Shackleton Crater** at the lunar south pole. The crater’s permanently shadowed floor preserves vast ice deposits, while its rim‑top receives near‑continuous sunlight—perfect for a self‑sustaining megacity.
* **Population:** Roughly 12 million permanent residents—scientists, engineers, diplomats, artists, and families—from Earth, Mars, the orbital habitats, and the few indigenous “Lunars” who have adapted to low‑gravity life.

---

### Architecture & Layout

| District | Key Features | Function |
|----------|--------------|----------|
| **Aurelia‑Arcology** | 

## Image input

In [2]:
import ipywidgets
from IPython import display

uploader = ipywidgets.FileUpload(accept='.png', multiple=False)
display.display(uploader)

FileUpload(value={}, accept='.png', description='Upload')

In [3]:
import base64

# This is a memoryview
content_mv = uploader.data[0]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [ ]:
import os
import time
import base64
from langchain import chat_models, agents, messages

with open("image.png", "rb") as f:
    img_b64 = base64.b64encode(f.read()).decode()

# Initialize using the OpenAI provider with a custom base_url
llm = chat_models.init_chat_model(
    model="llava:latest",
    model_provider="openai",
    base_url="https://ollama.com/v1",
    api_key=os.environ["OLLAMA_API_KEY"],
    temperature=0
)

agent = agents.create_agent(model=llm)

multimodal_question = messages.HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image_url", "image_url": {
        "url":f"data:image/png;base64,{img_b64}"
    }}
])

start_time = time.time()
response = agent.invoke(input={
    "messages": [multimodal_question]
})
end_time = time.time() - start_time
print(f"Time taken: %.2fs seconds"%(end_time))

for m in response['messages']:
    m.pretty_print()

## Audio input

In [ ]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

In [ ]:
agent = create_agent(
    model='gpt-4o-audio-preview',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)